# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 67df18d3-cf21-45b7-82c6-20647d3cced6
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session 67df18d3-cf21-45b7-82c6-20647d3cced6 to get into ready status...
Session 67df18d3-cf21-45b7-82c6-20647d3cced6 ha

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [2]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='customers')
dyf.printSchema()

root
|-- customer_id: string
|-- customer_unique_id: string
|-- customer_zip_code_prefix: long
|-- customer_city: string
|-- customer_state: string


#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [3]:
df = dyf.toDF()

df_sample = df.limit(1000)
df_sample.show()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

In [4]:
# Since the schema inferred by the AWS Glue crawler is correct, 
# I skip using the apply_mapping function and move directly with cleaning the data

# Step 1: Remove rows with duplicate customer_unique_id values
cleaned_df = df_sample.dropDuplicates(["customer_unique_id"])

# Step 2: Standardize city names
from pyspark.sql.functions import lower, trim, col

cleaned_df = cleaned_df.withColumn("customer_city", lower(trim(col("customer_city"))))

# Step 3: Fill missing customer_zip_code_prefix values with 0
cleaned_df = cleaned_df.na.fill(0, subset=["customer_zip_code_prefix"])

# Step 4: Validate customer_state codes with the official list for Brazil state codes
from pyspark.sql.functions import upper

brazil_codes = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG",
               "PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]

cleaned_df = cleaned_df.withColumn("customer_state", upper(col("customer_state")))

cleaned_df = cleaned_df.filter(upper(col("customer_state")).isin(brazil_codes))

In [5]:
# Convert the pyspark Dataframe back into a DynamicFrame
from awsglue.dynamicframe import DynamicFrame

cleaned_dyf = DynamicFrame.fromDF(cleaned_df, glueContext, "customers_dyf")

In [6]:
# Store the cleaned dataset as Parquet in S3
s3output = glueContext.getSink(
  path="s3://bucket181rt2/clean/customers",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="brazilian_ecommerce_clean", catalogTableName="customers"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(cleaned_dyf)